In [1]:
import sys
print(sys.executable)

/usr/bin/python3


In [2]:
import warp as wp
wp.init()
print(wp.get_device())  # Should print: cuda:0

Warp 1.12.0 initialized:
   CUDA Toolkit 12.9, Driver 13.1
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA GeForce RTX 4070" (12 GiB, sm_89, mempool enabled)
   Kernel cache:
     /home/sudharshan/.cache/warp/1.12.0
cuda:0


In [3]:
import warp as wp
import numpy as np
wp.init()

@wp.kernel
def apply_gravity(
    positions: wp.array(dtype=wp.vec3),
    velocities: wp.array(dtype=wp.vec3),
    dt: float
):
    i = wp.tid()
    if i == 0:                          # ✅ skip node 0 entirely
        return
    gravity = wp.vec3(0.0, -9.8, 0.0)
    velocities[i] = velocities[i] + gravity * dt
    positions[i]  = positions[i]  + velocities[i] * dt

n = 10
pos = wp.array(np.array([[0, i * 0.1, 0] for i in range(n)],
               dtype=np.float32), dtype=wp.vec3, device="cuda")
vel = wp.zeros(n, dtype=wp.vec3, device="cuda")

for _ in range(100):
    wp.launch(apply_gravity, dim=n, inputs=[pos, vel, 0.016])

print(pos.numpy())


Module __main__ efc1ff2 load on device 'cuda:0' took 2.05 ms  (cached)
[[  0.         0.         0.      ]
 [  0.       -12.56944    0.      ]
 [  0.       -12.469441   0.      ]
 [  0.       -12.369441   0.      ]
 [  0.       -12.269441   0.      ]
 [  0.       -12.16944    0.      ]
 [  0.       -12.069441   0.      ]
 [  0.       -11.969441   0.      ]
 [  0.       -11.869442   0.      ]
 [  0.       -11.769442   0.      ]]


In [4]:
import warp as wp

wp.init()

device = wp.get_device()
print(f"Device: {device}")           # Should print: cuda:0
print(f"GPU name: {device.name}")    # Should print: NVIDIA GeForce RTX 4070
print("Warp is working correctly!")

Device: cuda:0
GPU name: NVIDIA GeForce RTX 4070
Warp is working correctly!
